In [ ]:
!pip install fitz
!pip install PyMuPDF

In [ ]:
import os
import boto3
import json
from typing import Dict,Any

bedrock = boto3.client(service_name='bedrock-runtime')

def prepare_input(query,model_kwargs):
    input_body = {"anthropic_version": "bedrock-2023-05-31"}
    
    if 'max_tokens' in model_kwargs.keys() and int(model_kwargs['max_tokens']) > 0:
        input_body['max_tokens'] = model_kwargs['max_tokens']
        
    if 'temperature' in model_kwargs.keys() and float(model_kwargs['temperature']) > 0:
        input_body['temperature'] = model_kwargs['temperature']
        
    if 'system' in model_kwargs.keys() and len(model_kwargs['system']) > 0:
        input_body['system'] = model_kwargs['system']
        
    input_body['messages'] = []
    messages = {}
    messages['role'] = 'user'
    messages['content'] = []
    
    if len(query) > 0:
        text_dic = {"type":"text"}
        text_dic["text"] = query
        messages['content'].append(text_dic)

    if 'image' in model_kwargs.keys():
        image_dic = {"type": "image"}
        source = {"type": "base64","media_type": "image/jpeg"}
        source["data"] = model_kwargs['image']
        image_dic["source"] = source
        messages["content"].append(image_dic)
    
    input_body['messages'].append(messages)
    
    return input_body
    
def prepare_output(response):
    response_body = json.loads(response.get("body").read().decode())
    if 'content' in response_body.keys():
        return response_body.get("content")[0].get("text")
    else:
        return response_body.get("completion")

def invoke_claude_model(query: str,model_id:str,model_kwargs: Dict[str, Any]):
    
    modle_input = prepare_input(query,model_kwargs)
    body = json.dumps(modle_input)
    
    try:
        accept = "application/json"
        contentType = "application/json"
        response = bedrock.invoke_model(
            body=body, modelId=model_id, accept=accept, contentType=contentType
        )
        output = prepare_output(response)
        return output
    
    except Exception as e:
        raise ValueError(f"Error raised by bedrock service: {e}")

In [ ]:
from tqdm import tqdm
import fitz
from PIL import Image
import numpy as np
import base64
import io

# model_id = "anthropic.claude-3-5-sonnet-20240620-v1:0"
model_id = "anthropic.claude-3-sonnet-20240229-v1:0"

model_kwargs = {}
model_kwargs['temperature'] = 0.1
model_kwargs['max_tokens'] = 2048

image_max_size = 1024

prompt = """
You are a document manager at an financial company and your task is to extract useful information from document images.
<instructions>
1.don't make up content.
2.No preface, just output the document content directly.
3.Output the document in markdown format, and keep the rows and columns aligned for the table.
4.summarize page content to facilitate searching, output the summarize content in<summarize></summarize> tag
</instructions>
"""

# put the pdf in the files_path folder
files_path = './docs/'
files = os.listdir(files_path)
for file in files:
    file_path = files_path + file
    print(file_path)
    
    doc = fitz.open(file_path)
    for i in tqdm(range(doc.page_count)):
        page = doc.load_page(i)
        pix = page.get_pixmap(dpi=150)
                
        image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        print('image size:',image.size)
        # if image.size[0] > image_max_size:
        #     imgByteArr = io.BytesIO()
        #     image.save(imgByteArr,format='JPEG')
        #     image_bytes = imgByteArr.getvalue()
        # else:
        image_bytes = base64.b64encode(pix.tobytes()).decode("utf-8")
        model_kwargs['image'] = image_bytes
        
        
        response = invoke_claude_model(prompt,model_id,model_kwargs)     
        print(response)
        break
        
        
        
        